
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Create and share tables in Unity Catalog

In this notebook you will learn how to:
* Create schemas and tables
* Control access to schemas and tables
* Explore grants on various objects in Unity Catalog

## Set Up

Run the following cells to perform some setup. 

In order to avoid conflicts in a shared training environment, this will generate a unique catalog name exclusively for your use. 

In your own environment you are free to choose your own catalog names, but be careful about affecting other users & systems in that environment.

In [0]:
%run ./Includes/Classroom-Setup-06.2

## Unity Catalog three-level namespace

Most SQL developers will be familiar with using a two-level namespace to unambiguously address tables within a schema as follows:

    SELECT * FROM schema.table;

Unity Catalog introduces the concept of a *catalog* that resides above the schema in the object hierarchy. Metastores can host any number of catalogs, which in turn can host any number of schemas. To deal with this additional level, complete table references in Unity Catalog use a three-level namespace. The following statement exemplifies this:

    SELECT * FROM catalog.schema.table;
    
SQL developers will probably also be familiar with the **`USE`** statement to select a default schema, to avoid having to always specify a schema when referencing tables. Unity Catalog augments this with the **`USE CATALOG`** statement, which similarly selects a default catalog.

To simplify your experience, we ensured the catalog was created and set it as the default as you can see in the following command.

In [0]:
SELECT current_catalog(), current_database()

## Create and use a new schema

Let's create a new schema exclusively for our use in this exercise, then set this as the default so we can reference tables by name only.

In [0]:
CREATE SCHEMA IF NOT EXISTS my_own_schema;
USE my_own_schema;

SELECT current_database()

## Create Delta architecture

Let's create and populate a simple collection of schemas and tables pursuant to the Delta architecture:
* A silver schema containing patient heart rate data as read from a medical device
* A gold schema table that averages heart rate data per patient on a daily basis

For now, there will be no bronze table in this simple example.

Note that we need ony specify the table name below, since we have set a default catalog and schema above.

In [0]:
CREATE SCHEMA IF NOT EXISTS patient_silver;

CREATE OR REPLACE TABLE patient_silver.heartrate (
  device_id  INT,
  mrn        STRING,
  name       STRING,
  time       TIMESTAMP,
  heartrate  DOUBLE
);

INSERT INTO patient_silver.heartrate VALUES
  (23,'40580129','Nicholas Spears','2020-02-01T00:01:58.000+0000',54.0122153343),
  (17,'52804177','Lynn Russell','2020-02-01T00:02:55.000+0000',92.5136468131),
  (37,'65300842','Samuel Hughes','2020-02-01T00:08:58.000+0000',52.1354807863),
  (23,'40580129','Nicholas Spears','2020-02-01T00:16:51.000+0000',54.6477014191),
  (17,'52804177','Lynn Russell','2020-02-01T00:18:08.000+0000',95.033344842),
  (37,'65300842','Samuel Hughes','2020-02-01T00:23:58.000+0000',57.3391541312),
  (23,'40580129','Nicholas Spears','2020-02-01T00:31:58.000+0000',56.6165053697),
  (17,'52804177','Lynn Russell','2020-02-01T00:32:56.000+0000',94.8134313932),
  (37,'65300842','Samuel Hughes','2020-02-01T00:38:54.000+0000',56.2469995332),
  (23,'40580129','Nicholas Spears','2020-02-01T00:46:57.000+0000',54.8372685558)

In [0]:
CREATE SCHEMA IF NOT EXISTS patient_gold;

CREATE OR REPLACE TABLE patient_gold.heartrate_stats AS (
  SELECT mrn, name, MEAN(heartrate) avg_heartrate, DATE_TRUNC("DD", time) date
  FROM patient_silver.heartrate
  GROUP BY mrn, name, DATE_TRUNC("DD", time)
);
  
SELECT * FROM patient_gold.heartrate_stats;

## Grant access to gold schema [optional]

Now let's allow users in the **account users** group to read from the **gold** schema.

Perform this section by uncommenting the code cells and running them in sequence. 
You will also be prompted to run some queries. 

To do this:
1. Open a separate browser tab and load your Databricks workspace.
1. Switch to Databricks SQL by clicking on the app switcher and selecting SQL.
1. Create a SQL warehouse following the instructions in *Create SQL Warehouse in Unity Catalog*.
1. Prepare to enter queries as instructed below in that environment.

Let's grant **SELECT** privilege on the **gold** table.

In [0]:
-- GRANT SELECT ON TABLE patient_gold.heartrate_stats to `account users`

### Query table as user

With a **SELECT** grant in place, attempt to query the table in the Databricks SQL environment.

Run the following cell to output a query statement that reads from the **gold** table. Copy and paste the output into a new query within the SQL environment and run the query.

In [0]:
%python
print(f"SELECT * FROM {DA.catalog_name}.patient_gold.heartrate_stats")

This command works for us since we are the owner of the view. However, it will not work yet for other members of the **account users** group because **SELECT** privilege on the table alone is insufficient. **USAGE** privilege is also required on the containing elements. Let's correct this now by executing the following.

In [0]:
-- GRANT USAGE ON CATALOG ${DA.catalog_name} TO `account users`;
-- GRANT USAGE ON SCHEMA patient_gold TO `account users`


Repeat the query in the Databricks SQL environment, and with these two grants in place the operation should succeed.


## Explore grants

Let's explore the grants on some of the objects in the Unity Catalog hierarchy, starting with the **gold** table.

In [0]:
-- SHOW GRANT ON TABLE ${DA.catalog_name}.patient_gold.heartrate_stats

Currently there is only the **SELECT** grant we set up earlier. Now let's check the grants on **silver**.

In [0]:
SHOW TABLES IN ${DA.catalog_name}.patient_silver;

In [0]:
-- SHOW GRANT ON TABLE ${DA.catalog_name}.patient_silver.heartrate

There are currently no grants on this table; only the owner can access this table.

Now let's look at the containing schema.

In [0]:
-- SHOW GRANT ON SCHEMA ${DA.catalog_name}.patient_silver

There are currently no grants on this schema. 

Now let's examine the catalog.

In [0]:
-- SHOW GRANT ON CATALOG `${DA.catalog_name}`

Currently we see the **USAGE** grant we set up earlier.

## Clean up
Run the following cell to remove the schema that we created in this example.

In [0]:
%python
DA.cleanup()


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>